# General guidelines

- Refer to the notebook of the tutorial session taken on 15th October for implementation of following problems
- Download a copy of this notebook and perform your implementations here.
- Submit this notebook after implementation using the following format: firstname_srNO(last 5 digits).pynb
eg. mahesh_20369.ipynb

# Q1

## Objective


In this assignment, you will design and implement a lightweight LLM-powered agent that interacts with an Airbnb-like server API using structured data models.
Your agent will be responsible for verifying the validity of links returned by the Airbnb endpoints and ensuring that only working URLs appear in the final output.



## Q1 Guidelines

All students must use a common lightweight language model to ensure fairness in evaluation (e.g., gpt-4o-mini, Mistral-7B-Instruct, Phi-3-mini, or Llama 3 8B Instruct).
You may, however, experiment with:

- System prompts

- Verifier agent logic

### Functional Requirements

You are required to implement three functions that interact with the Airbnb server:

- get_current_time(params: dict)
Returns the current time from the Airbnb server.

- airbnb_search(params: dict)
Performs a search query (e.g., by location, check-in date, price range) on the Airbnb server.

- airbnb_listing_details(params: dict)
Fetches detailed information for a specific listing.

Each function must produce structured, validated output using Pydantic models.

### Define Structured Data Models

Use Pydantic to define schemas for:

- Current time response

- Search results (with listing summaries)

- Listing details

Each schema should validate key fields such as IDs, titles, prices, amenities, and URLs.

### Implement Airbnb API Wrappers

Each of the three required functions will call the respective Airbnb API endpoints and return JSON responses that are parsed and validated against your Pydantic models.
You may connect to a provided Airbnb mock server or simulate your own endpoints.

### Recommended Libraries

You may use the following Python libraries:

- Pydantic → Schema validation

- Requests → HTTP calls and link verification

- Tenacity → Retry mechanism for failed link checks

- Transformers or OpenAI API → Lightweight LLM interface


FastAPI / Flask (optional) → To simulate a local Airbnb mock API









In [1]:
# you should dedicate a code cell next to the markdown cell "OUTPUT" to produce a final JSON output of the following form



{
  "current_time": "YYYY-MM-DDTHH:MM:SSZ",
  "search_results": [
    {
      "id": "A101",
      "title": "Cozy apartment in Paris",
      "url": "https://www.airbnb.com/rooms/12345",
      "price_per_night": 120.0
    }
  ],
  "verified_links": [
    "https://www.airbnb.com/rooms/12345"
  ]
}




{'current_time': 'YYYY-MM-DDTHH:MM:SSZ',
 'search_results': [{'id': 'A101',
   'title': 'Cozy apartment in Paris',
   'url': 'https://www.airbnb.com/rooms/12345',
   'price_per_night': 120.0}],
 'verified_links': ['https://www.airbnb.com/rooms/12345']}

## Evaluation Criteria

| **Component**                      | **Points** | **Description**                                                      |
| ---------------------------------- | ---------- | -------------------------------------------------------------------- |
| Pydantic model definitions         | 20         | Models correctly represent response structure and enforce validation |
| Airbnb API wrapper functions       | 20         | Functions correctly fetch and return structured data                 |
| Verifier agent implementation      | 25         | Accurately checks and filters out invalid URLs                       |
| Integration with LLM               | 15         | LLM used effectively for reasoning or verification                   |
| Output format and reproducibility  | 10         | JSON output correctly structured and reproducible                    |
| Prompt design and agent creativity | 10         | Quality and clarity of prompt/system design                          |


## Output

In [2]:
%%capture
!pip install pydantic_ai openai

### SERVER (mock FastAPI)
Paste this in a separate file and run 'python airbnb_server.py'

In [3]:
# airbnb_server.py
'''
import uvicorn
from fastapi import FastAPI
from datetime import datetime
import random

store = {
  "listing_1": {
    "price_per_night": 120,
    "rating": 3,
    "title": "Cozy Apartment",
    "url": "https://www.airbnb.com/rooms/listing_1"
  },
  "listing_2": {
    "price_per_night": 250,
    "title": "Beach House",
    "url": "https://www.airbnb.com/rooms/listing_2"
  }
}
location = "Did not get updated"

app = FastAPI()


@app.post("/get_current_time")
async def get_current_time(params: dict):
  return {
    "server_time": datetime.utcnow().isoformat(),
    "params_received": params,
    "status": "Success",
    "error": ""
  }


@app.post("/airbnb_search")
async def airbnb_search(params: dict):

  if "location" not in params:
    return {
      "total_results": 0,
      "listings": [],
      "params_received": params,
      "status": "Failure",
      "error": "Location is Required"
    }
  global location
  location = params.get("location")

  # mock result list
  mock_listings = []
  for key, val in store.items():
    new_val = val.copy()
    new_val["id"] = key
    new_val["location"] = location
    mock_listings.append(new_val)

  return {
      "total_results": len(mock_listings),
      "listings": mock_listings,
      "params_received": params,
      "status": "Success",
      "error": ""
    }


@app.post("/airbnb_listing_details")
async def airbnb_listing_details(params: dict):

  if "id" not in params:
    return {
      "id": params.get("id", "unknown"),
      "title": "",
      "description": "",
      "price_per_night": 0,
      "rating": 0,
      "amenities": [],
      "host": "",
      "location": "",
      "params_received": params,
      "status": "Failure",
      "error": "id is required"
    }
  amenities = ["WiFi", "Kitchen", "Air Conditioning"]
  new_val = store.get(params.get("id"))
  new_val["id"] = params.get("id")
  new_val["location"] = location
  new_val["description"] = "This is a mock listing description."
  new_val["host"] = "Mock Host"
  new_val["amenities"] = [random.choice(amenities)]

  result = {
    "params_received": params,
    "status": "Success",
    "error": ""
  }
  result.update(new_val)

  # mock details
  return result

if __name__ == "__main__":
  uvicorn.run(app, host="127.0.0.1", port=8000)
  '''

'\nimport uvicorn\nfrom fastapi import FastAPI\nfrom datetime import datetime\nimport random\n\nstore = {\n  "listing_1": {\n    "price_per_night": 120,\n    "rating": 3,\n    "title": "Cozy Apartment",\n    "url": "https://www.airbnb.com/rooms/listing_1"\n  },\n  "listing_2": {\n    "price_per_night": 250,\n    "title": "Beach House",\n    "url": "https://www.airbnb.com/rooms/listing_2"\n  }\n}\nlocation = "Did not get updated"\n\napp = FastAPI()\n\n\n@app.post("/get_current_time")\nasync def get_current_time(params: dict):\n  return {\n    "server_time": datetime.utcnow().isoformat(),\n    "params_received": params,\n    "status": "Success",\n    "error": ""\n  }\n\n\n@app.post("/airbnb_search")\nasync def airbnb_search(params: dict):\n  \n  if "location" not in params:\n    return {\n      "total_results": 0,\n      "listings": [],\n      "params_received": params,\n      "status": "Failure",\n      "error": "Location is Required"\n    }\n  global location\n  location = params.get("

### Question 1

In [4]:
# PYDANTIC SCHEMAS
from pydantic import BaseModel, Field, HttpUrl, field_validator
from typing import List, Optional
from datetime import datetime

# 'get_current_time' response schema
class CurrentTimeResponse(BaseModel):
    params_received: dict = Field(..., description="Input search parameters")
    server_time: datetime = Field(..., description="Current server timestamp")
    timezone: str = Field(..., description="Timezone string (e.g., UTC, PST)")

class ListingSummary(BaseModel):
    id: str = Field(..., description="Unique listing ID")
    title: str = Field(..., min_length=1)
    price_per_night: float = Field(..., gt=0)
    url: HttpUrl
    rating: Optional[float] = Field(None, ge=0, le=5)
    location: str = Field(...)

    @field_validator("id")
    def id_not_empty(cls, v):
        if len(v.strip()) == 0:
            raise ValueError("Listing ID cannot be empty")
        return v

# 'airbnb_search' response schema
class SearchResultsResponse(BaseModel):
    params_received: dict = Field(..., description="Input search parameters")
    total_results: int = Field(..., ge=0)
    listings: List[ListingSummary]

# 'airbnb_listing_details'
class ListingDetails(BaseModel):
    id: str
    params_received: dict = Field(..., description="Input search parameters")
    title: str
    description: str
    price_per_night: float = Field(..., gt=0)
    amenities: List[str]
    host: str
    rating: Optional[float] = Field(None, ge=0, le=5)
    location: str

    @field_validator("amenities")
    def nonempty_amenities(cls, v):
        if not v:
            raise ValueError("Amenities list cannot be empty")
        return v

class ListingAll(BaseModel):
    id: str
    title: str
    price_per_night: float = Field(..., gt=0)
    amenities: List[str]
    host: str
    rating: Optional[float] = Field(None, ge=0, le=5)
    location: str
    url: HttpUrl

    @field_validator("amenities")
    def nonempty_amenities(cls, v):
        if not v:
            raise ValueError("Amenities list cannot be empty")
        return v

class ToolCall(BaseModel):
  tool_name: str
  params: dict

# Model output validation
class FinalOutput(BaseModel):
  tool_calls: List[ToolCall] = Field(..., description="All tool calls done by the LLM")
  current_time: datetime = Field(..., description="Current server timestamp")
  search_results: List[ListingAll] = Field(..., description="Listing details")
  verified_links: List[HttpUrl] = Field(..., description="List of valid links")

In [5]:
# API WRAPPER FUNCTIONS
import requests
from pydantic_ai import Tool
from pydantic import ValidationError

BASE_URL = "http://localhost:8000"

@Tool
def get_current_time(params: dict) -> CurrentTimeResponse:
  """Gets Current Time in UTC format
  Parameters:
    - params(dict): Any arguments to the API. Example: params={"timezone": "UTC"}

  Return:
    - CurrentTimeResponse

  Example :
  get_current_time({})
  """
  url = f"{BASE_URL}/get_current_time"
  #params = params.params
  # Call API and check for Failure
  r = requests.post(url, json=params, timeout=5).json()
  if r["status"] == "Failure":
    return ValueError(r["error"])
  del r["status"], r["error"]
  r["timezone"] = "UTC"
  # Validate output and return
  return CurrentTimeResponse(**r)

@Tool
def airbnb_search(params: dict) -> SearchResultsResponse:
  """Search for Airbnb Listing
  Parameters:
    - params(dict): Any arguments to the API. Example: params={"location": Required, ...}

  Return:
    - SearchResultsResponse
  """
  # Validate input
  #params = params.params
  if "location" not in params: return ValueError("Location is required.")
  url = f"{BASE_URL}/airbnb_search"
  # Call API and check for Failure
  r = requests.post(url, json=params, timeout=10).json()
  if r["status"] == "Failure":
    return ValueError(r["error"])
  del r["status"], r["error"]
  # Validate output and return
  return SearchResultsResponse(**r)

@Tool
def airbnb_listing_details(params: dict) -> ListingDetails:
  """Get Details for a specific airbnb
  Parameters:
    - params(dict): Any arguments to the API. Example: params={"id": Required, ...}

  Return:
    - ListingDetails
  """
  # Validate input
  #params = params.params
  if "id" not in params: return ValueError("ID is required")
  url = f"{BASE_URL}/airbnb_listing_details"
  # Call API and check for Failure
  r = requests.post(url, json=params, timeout=10).json()
  if r["status"] == "Failure":
    return ValueError(r["error"])
  del r["status"], r["error"]
  # Validate output and return
  return ListingDetails(**r)

In [6]:
# List all available models incase rate limit reached
# from openai import OpenAI
# openai_api_key = ""
# client = OpenAI(api_key=openai_api_key)

# models = client.models.list()
# for m in models.data:
#   if "gpt" in m.id or "text" in m.id:
#     print(m.id)

In [7]:
# use this cell to generate output for Q1.
import json
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent

# Make the Agent
model_id = "gpt-4.1-mini" #"gpt-4o-mini"
openai_api_key = ""
openai_api_base = "https://api.openai.com/v1"

provider = OpenAIProvider(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
agent_model = OpenAIChatModel(model_id, provider=provider)
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)

When calling any tool, you must pass all inputs inside a dict named 'params'.
For example, to search for listings, always call:

airbnb_search(params={
    "location": "<city>",
    "check_in": "<YYYY-MM-DD>",
    "check_out": "<YYYY-MM-DD>",
    "adults": 2
})

Do NOT pass any arguments outside 'params'.

When the user asks for listings,
first call get_current_time,
then airbnb_search,
then airbnb_listing_details using id from search,
etc.

DO NOT ANY STRING, DIRECTLY GIVE THE JSON OUTPUT
Make sure the output is in the following JSON format

{
  "tool_calls": [
    {
      "tool_name": "Tool 1",
      "params": params
    }
  ]
  "current_time": "YYYY-MM-DDTHH:MM:SSZ",
  "search_results": [
    {
      "id": "A101",
      "title": "Cozy apartment in Paris",
      "url": "https://www.airbnb.com/rooms/12345",
      "price_per_night": 120.0,
      "location": "Paris",
      "rating": 0-5 | Optional,
      "amenities": ["WiFi", "Kitchen", ...],
      "host": "Mock Host"
    }
  ],
  "verified_links": [
    "https://www.airbnb.com/rooms/12345"
  ]
}
"""
tools = [get_current_time, airbnb_search, airbnb_listing_details]

agent = Agent(
    model=agent_model,
    tools=tools,
    system_prompt=system_prompt,
    output_type=FinalOutput
    )

# Interaction function
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output

result = await run_async("Find a place to stay in Jammu for next Sunday for 3 nights for 2 adults?")

format_input = result.model_dump_json()
format_input = json.loads(format_input)
print(f"Tools called by listing extraction agent:")
for tool in format_input["tool_calls"]:
  print(f"{tool['tool_name']}: {tool['params']}")
del format_input["tool_calls"]

print("Final output to be made input to next agent:")

print(f"{format_input}")

Tools called by listing extraction agent:
get_current_time: {}
airbnb_search: {'location': 'Jammu', 'check_in': '2025-12-07', 'check_out': '2025-12-10', 'adults': 2}
airbnb_listing_details: {'id': 'listing_1'}
airbnb_listing_details: {'id': 'listing_2'}
Final output to be made input to next agent:
{'current_time': '2025-11-29T21:27:05Z', 'search_results': [{'id': 'listing_1', 'title': 'Cozy Apartment', 'price_per_night': 120.0, 'amenities': ['WiFi'], 'host': 'Mock Host', 'rating': 3.0, 'location': 'Jammu', 'url': 'https://www.airbnb.com/rooms/listing_1'}, {'id': 'listing_2', 'title': 'Beach House', 'price_per_night': 250.0, 'amenities': ['Air Conditioning'], 'host': 'Mock Host', 'rating': None, 'location': 'Jammu', 'url': 'https://www.airbnb.com/rooms/listing_2'}], 'verified_links': ['https://www.airbnb.com/rooms/listing_1', 'https://www.airbnb.com/rooms/listing_2']}


# Q2

## Objective

Using the verified JSON output generated in Question 1, design an LLM-based module that parses and reformats the data into a clean, human-readable list of Airbnb listings.
Unlike Question 1, this stage does not perform link verification — it assumes all links are already valid.

Your goal is to:

- Feed the verified JSON back to the LLM,

- Prompt it to extract only the key user-facing details, and

- Produce a clean, structured list or summary that can be directly displayed to end users or used by downstream systems


## Q2 Guidelines


### 1. Load the Verified JSON

Load the JSON object produced in Question 1 into your notebook.
Ensure your code dynamically reads the keys rather than hard-coding them.

### 2. Design an LLM Prompt

Construct a prompt that:

- Accepts the JSON as context.

- Instructs the model to return a clean, concise list summarizing each listing’s:

  - Title

  - Price per night

  - URL

- Optionally, includes additional natural-language formatting (e.g., numbered list, short summary lines).

- Explicitly forbids re-verifying or modifying URLs.


### 3. Ensure the LLM response:

Matches all entries in the verified list.

Contains no hallucinated or missing items.

Requires no further link verification.








## Evaluation Criteria

| **Component**                         | **Points** | **Description**                                                      |
| ------------------------------------- | ---------- | -------------------------------------------------------------------- |
| JSON ingestion & preprocessing        | 10         | Correctly loads and handles verified JSON                            |
| Prompt construction                   | 10         | Clear and robust system/user prompt design                           |
| Clean, correctly formatted LLM output | 20         | Output includes all valid listings, no verification or hallucination |
| Reproducibility & LLM configuration   | 10         | Uses consistent lightweight model with documented parameters         |




In [8]:
from pydantic_ai import Agent
from IPython.display import Markdown, display

# Make the Agent
system_prompt = """
You are a natural language converter agent.
You will be given a some json string.

You need to convert it into human readable format.
Add bold titles, numbered bullets, summaries, and whatever extra stuff you feel necessary.

Only give key user facing details, nothing extra.
DO NOT MODIFY THE PROVIDED LINKS or DATA. USE ALL THE DATA GIVEN.
"""

final_agent = Agent(
    model=agent_model,
    system_prompt=system_prompt,
    )

# Interaction function
async def run_async(prompt: str) -> str:
    async with final_agent.run_mcp_servers():
        result = await final_agent.run(prompt)
        return result.output

output = await run_async(f"{format_input}")
print("Raw Output:")
print("=======================================================================")
print(output)

Raw Output:
**Airbnb Listings Summary (As of 2025-11-29 21:27 UTC)**

Here are two accommodation options available in Jammu:

1. **Cozy Apartment**
   - Price: $120 per night
   - Amenities: WiFi
   - Host: Mock Host
   - Rating: 3.0 stars
   - Location: Jammu
   - Booking Link: [Cozy Apartment](https://www.airbnb.com/rooms/listing_1)

2. **Beach House**
   - Price: $250 per night
   - Amenities: Air Conditioning
   - Host: Mock Host
   - Rating: Not available
   - Location: Jammu
   - Booking Link: [Beach House](https://www.airbnb.com/rooms/listing_2)

**Summary:**  
Choose the Cozy Apartment for an affordable stay with WiFi and a moderate rating. If you prefer a more premium option with air conditioning, the Beach House is available but currently has no rating. Both listings are verified.


In [9]:
display(Markdown(output))

**Airbnb Listings Summary (As of 2025-11-29 21:27 UTC)**

Here are two accommodation options available in Jammu:

1. **Cozy Apartment**
   - Price: $120 per night
   - Amenities: WiFi
   - Host: Mock Host
   - Rating: 3.0 stars
   - Location: Jammu
   - Booking Link: [Cozy Apartment](https://www.airbnb.com/rooms/listing_1)

2. **Beach House**
   - Price: $250 per night
   - Amenities: Air Conditioning
   - Host: Mock Host
   - Rating: Not available
   - Location: Jammu
   - Booking Link: [Beach House](https://www.airbnb.com/rooms/listing_2)

**Summary:**  
Choose the Cozy Apartment for an affordable stay with WiFi and a moderate rating. If you prefer a more premium option with air conditioning, the Beach House is available but currently has no rating. Both listings are verified.